# Induction Head Exhaustive Causal Search

Exhaustive zero-ablation causal sweep across all 384 attention heads (24 layers × 16 heads) in SantaCoder on synthetic repeated-token prompts.

Results are averaged across 3 seeds. Final output: `induction_exhaustive_results.csv`.

In [1]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformer_lens==2.17.0",
    "transformers==4.57.6",
    "einops==0.8.2",
    "typeguard==4.5.1",
], check=True)

# Patch SantaCoder compatibility
import os, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

_REPO_ID = "bigcode/santacoder"
_PATCHES = {
    "modeling_gpt2_mq.py": [
        (
            "from transformers.modeling_utils import PreTrainedModel, SequenceSummary",
            "from transformers.modeling_utils import PreTrainedModel",
        ),
    ],
}

def patch_file(path, replacements):
    content = path.read_text()
    for target, replacement in replacements:
        content = content.replace(target, replacement)
    path.write_text(content)
    print(f"patched: {path.name}")

for filename, replacements in _PATCHES.items():
    snapshot_path = Path(hf_hub_download(_REPO_ID, filename))
    patch_file(snapshot_path, replacements)

dyn_cache = Path.home() / ".cache" / "huggingface" / "modules" / "transformers_modules" / "bigcode"
if dyn_cache.exists():
    shutil.rmtree(dyn_cache)
    print(f"cleared dynamic cache: {dyn_cache}")

print("Setup complete. Now go to Runtime → Restart session, then run all cells below this one.")

/home/estevao/mechinterp/.venv/bin/python3: No module named pip


CalledProcessError: Command '['/home/estevao/mechinterp/.venv/bin/python3', '-m', 'pip', 'install', '-q', 'transformer_lens', 'einops']' returned non-zero exit status 1.

In [3]:
import torch
from transformer_lens import HookedTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

model = HookedTransformer.from_pretrained(
    "santacoder",
    device=DEVICE,
)
model.cfg.use_attn_result = True
model.eval()

N_LAYERS = model.cfg.n_layers  # 24
N_HEADS  = model.cfg.n_heads   # 16
print(f"n_layers={N_LAYERS}, n_heads={N_HEADS}")

Using device: cpu


`torch_dtype` is deprecated! Use `dtype` instead!


TypeError: GPT2LMHeadCustomModel.__init__() got an unexpected keyword argument 'use_attn_result'

In [ ]:
import re
from dataclasses import dataclass

SAFE_TOKEN_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]{1,11}$")

@dataclass
class PromptExample:
    repeated: list
    control: list
    target: int
    detection_sequence: list
    align_pairs: list  # corrected induction metric: (seq_len+i, i+1)

def safe_token_pool(pool_size: int = 512) -> list:
    seen, ids = set(), []
    for tok_id in range(model.cfg.d_vocab):
        try:
            tok = model.to_single_str_token(tok_id).strip()
        except Exception:
            continue
        if not SAFE_TOKEN_RE.fullmatch(tok):
            continue
        low = tok.lower()
        if low in seen:
            continue
        seen.add(low)
        ids.append(tok_id)
        if len(ids) >= pool_size:
            break
    assert len(ids) >= 64, f"Only {len(ids)} safe tokens found"
    return ids

def build_dataset(n_prompts: int, seq_len: int, seed: int) -> list:
    torch.manual_seed(seed)
    pool = safe_token_pool(pool_size=max(512, n_prompts * seq_len * 4))
    g = torch.Generator().manual_seed(seed)
    dataset = []
    for _ in range(n_prompts):
        perm = torch.randperm(len(pool), generator=g).tolist()
        primary    = [pool[i] for i in perm[:seq_len]]
        distractor = [pool[i] for i in perm[seq_len: 2 * seq_len]]
        dataset.append(PromptExample(
            repeated=primary + primary[:-1],          # length 2*seq_len - 1
            control=distractor + primary[:-1],
            target=primary[-1],
            detection_sequence=primary + primary,     # length 2*seq_len
            # corrected induction alignment: 2nd copy of t_i attends to t_{i+1}
            align_pairs=[(seq_len + i, i + 1) for i in range(seq_len - 1)],
        ))
    return dataset

# Quick sanity check
ex = build_dataset(n_prompts=2, seq_len=8, seed=0)[0]
print("repeated length:", len(ex.repeated))
print("detection_sequence length:", len(ex.detection_sequence))
print("align_pairs:", ex.align_pairs)

In [ ]:
# Corrected detection metric (Olsson et al. 2022):
# For each example, score each head by mean attention from (seq_len+i) -> (i+1)

@torch.no_grad()
def score_detection(dataset: list) -> torch.Tensor:
    """Returns [n_layers, n_heads] mean detection scores."""
    scores = torch.zeros(N_LAYERS, N_HEADS, device=DEVICE)
    for ex in dataset:
        tokens = torch.tensor([ex.detection_sequence], device=DEVICE)
        _, cache = model.run_with_cache(tokens)
        pairs = torch.tensor(ex.align_pairs, device=DEVICE)  # [n_pairs, 2]
        q_idx = pairs[:, 0]
        k_idx = pairs[:, 1]
        for layer in range(N_LAYERS):
            pattern = cache["pattern", layer][0]  # [n_heads, seq, seq]
            scores[layer] += pattern[:, q_idx, k_idx].mean(dim=-1)
    return scores / len(dataset)

In [ ]:
# Zero-ablation via hook_result (head output before summing into residual stream)

@torch.no_grad()
def mean_logprob(dataset: list, kind: str,
                 ablate_layer: int = None, ablate_head: int = None) -> float:
    hooks = []
    if ablate_layer is not None:
        hook_name = f"blocks.{ablate_layer}.attn.hook_result"
        def _zero(z, hook, h=ablate_head):
            z = z.clone()
            z[:, :, h, :] = 0.0
            return z
        hooks = [(hook_name, _zero)]

    total = 0.0
    for ex in dataset:
        tokens = torch.tensor([getattr(ex, kind)], device=DEVICE)
        target = torch.tensor([[ex.target]], device=DEVICE)
        logits = model.run_with_hooks(tokens, fwd_hooks=hooks) if hooks else model(tokens)
        lp = logits[:, -1].log_softmax(-1).gather(-1, target).squeeze()
        total += float(lp.item())
    return total / len(dataset)

print("Causal ablation function defined.")

In [ ]:
import numpy as np
from tqdm.auto import tqdm

N_PROMPTS = 25
SEQ_LEN   = 8
SEEDS     = [0, 1, 2]

# results[layer][head] -> list of values per seed
all_repeated_drops  = [[[] for _ in range(N_HEADS)] for _ in range(N_LAYERS)]
all_control_drops   = [[[] for _ in range(N_HEADS)] for _ in range(N_LAYERS)]
all_detection_scores = [[[] for _ in range(N_HEADS)] for _ in range(N_LAYERS)]

for seed in SEEDS:
    print(f"\n=== Seed {seed} ===")
    dataset = build_dataset(n_prompts=N_PROMPTS, seq_len=SEQ_LEN, seed=seed)

    baseline_rep = mean_logprob(dataset, "repeated")
    baseline_ctl = mean_logprob(dataset, "control")
    print(f"  baseline repeated={baseline_rep:.4f}  control={baseline_ctl:.4f}  "
          f"gain={baseline_rep - baseline_ctl:.4f}")

    detection_scores = score_detection(dataset)  # [n_layers, n_heads]

    for layer in range(N_LAYERS):
        for head in tqdm(range(N_HEADS), desc=f"  layer {layer:02d}", leave=False):
            abl_rep = mean_logprob(dataset, "repeated", ablate_layer=layer, ablate_head=head)
            abl_ctl = mean_logprob(dataset, "control",  ablate_layer=layer, ablate_head=head)
            all_repeated_drops[layer][head].append(baseline_rep - abl_rep)
            all_control_drops[layer][head].append(baseline_ctl - abl_ctl)
            all_detection_scores[layer][head].append(float(detection_scores[layer, head].item()))
        print(f"  layer {layer:02d} done")

print("\nSweep complete.")

In [ ]:
import pandas as pd

rows = []
for layer in range(N_LAYERS):
    for head in range(N_HEADS):
        rep_drops = np.array(all_repeated_drops[layer][head])
        ctl_drops = np.array(all_control_drops[layer][head])
        sel_drops = rep_drops - ctl_drops
        det = np.array(all_detection_scores[layer][head])
        rows.append({
            "layer": layer,
            "head": head,
            "mean_causal_selective_drop": float(sel_drops.mean()),
            "std_causal_selective_drop":  float(sel_drops.std()),
            "mean_repeated_drop": float(rep_drops.mean()),
            "mean_control_drop":  float(ctl_drops.mean()),
            "mean_detection_score": float(det.mean()),
            "std_detection_score":  float(det.std()),
        })

results_df = (
    pd.DataFrame(rows)
    .sort_values("mean_causal_selective_drop", ascending=False)
    .reset_index(drop=True)
)

print("Top-20 heads by mean causal selective drop:")
results_df.head(20)

In [ ]:
from google.colab import files

csv_path = "induction_exhaustive_results.csv"
results_df.to_csv(csv_path, index=False)
print(f"Saved {len(results_df)} rows to {csv_path}")
files.download(csv_path)